In [10]:
import pandas as pd
import numpy as np
import os

DATA_DIR = r"C:\\Users\\nguye\\Desktop\\datathon\\data"
OUT_DIR  = r"C:\\Users\\nguye\\Desktop\\datathon\\tableau_data"
os.makedirs(OUT_DIR, exist_ok=True)

def load(name, **kw):
    path = os.path.join(DATA_DIR, name)
    df = pd.read_csv(path, **kw)
    print(f"  Loaded {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df

def save(df, name):
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"  Saved {name}: {df.shape[0]:,} rows ({size_mb:.1f} MB)")

In [4]:
orders      = load('orders.csv',      parse_dates=['order_date'])
order_items = load('order_items.csv')
payments    = load('payments.csv')
products    = load('products.csv')
customers   = load('customers.csv',   parse_dates=['signup_date'])
geography   = load('geography.csv')
promotions  = load('promotions.csv',  parse_dates=['start_date', 'end_date'])
shipments   = load('shipments.csv',   parse_dates=['ship_date', 'delivery_date'])
returns     = load('returns.csv',     parse_dates=['return_date'])
reviews     = load('reviews.csv',     parse_dates=['review_date'])
inventory   = load('inventory.csv',   parse_dates=['snapshot_date'])
web_traffic = load('web_traffic.csv', parse_dates=['date'])
sales       = load('sales.csv',       parse_dates=['Date'])

  Loaded orders.csv: 646,945 rows x 8 cols


C:\Users\nguye\AppData\Local\Temp\ipykernel_18188\2348067705.py:11: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **kw)


  Loaded order_items.csv: 714,669 rows x 7 cols
  Loaded payments.csv: 646,945 rows x 4 cols
  Loaded products.csv: 2,412 rows x 8 cols
  Loaded customers.csv: 121,930 rows x 7 cols
  Loaded geography.csv: 39,948 rows x 4 cols
  Loaded promotions.csv: 50 rows x 10 cols
  Loaded shipments.csv: 566,067 rows x 4 cols
  Loaded returns.csv: 39,939 rows x 7 cols
  Loaded reviews.csv: 113,551 rows x 7 cols
  Loaded inventory.csv: 60,247 rows x 17 cols
  Loaded web_traffic.csv: 3,652 rows x 7 cols
  Loaded sales.csv: 3,833 rows x 3 cols


In [11]:
dim_products = products.copy()
dim_products['gross_margin_pct'] = (
    (dim_products['price'] - dim_products['cogs']) / dim_products['price'] * 100
).round(2)
dim_products['profit_per_unit'] = (dim_products['price'] - dim_products['cogs']).round(2)

# b) Join avg rating tu reviews
avg_rating = reviews.groupby('product_id')['rating'].agg(['mean', 'count']).reset_index()
avg_rating.columns = ['product_id', 'avg_rating', 'review_count']

# c) Join return stats tu returns
ret_stats = returns.groupby('product_id').agg(
    total_returns    = ('return_id', 'count'),
    total_return_qty = ('return_quantity', 'sum'),
    total_refund     = ('refund_amount', 'sum')
).reset_index()

dim_products = dim_products.merge(avg_rating, on='product_id', how='left')
dim_products = dim_products.merge(ret_stats,  on='product_id', how='left')

# d) Xu ly NaN
dim_products[['review_count', 'total_returns', 'total_return_qty']] = \
    dim_products[['review_count', 'total_returns', 'total_return_qty']].fillna(0).astype(int)
dim_products['total_refund'] = dim_products['total_refund'].fillna(0)

save(dim_products,"dim_products.csv")

  Saved dim_products.csv: 2,412 rows (0.3 MB)


In [12]:
save(geography,"dim_geography.csv")
save(promotions,"dim_promotions.csv")

  Saved dim_geography.csv: 39,948 rows (1.3 MB)
  Saved dim_promotions.csv: 50 rows (0.0 MB)


In [15]:
fact = order_items.copy()
fact = fact.merge(
orders[["order_id","order_date","customer_id","zip",
"order_status","payment_method","device_type","order_source"]],
on="order_id", how="left")
fact = fact.merge(
products[["product_id","product_name","category","segment",
"size","color","price","cogs"]],
on="product_id", how="left")
fact = fact.merge(
payments[["order_id","payment_value","installments"]],
on="order_id", how="left")
fact = fact.merge(
geography[["zip","city","region"]].drop_duplicates(subset="zip"),
on="zip", how="left")

fact["line_revenue"] = (fact["quantity"] * fact["unit_price"]).round(2)
fact["line_cost"] = (fact["quantity"] * fact["cogs"]).round(2)
fact["line_gross_profit"] = (fact["line_revenue"] - fact["line_cost"]).round(2)
fact["line_margin_pct"] = np.where(
fact["line_revenue"] > 0,
(fact["line_gross_profit"] / fact["line_revenue"] * 100).round(2), 0)
fact["has_promo"] = (~fact["promo_id"].isna() & (fact["promo_id"] != "")).astype(int
)
fact["discount_pct"] = np.where(
fact["line_revenue"] + fact["discount_amount"] > 0,
(fact["discount_amount"] / (fact["line_revenue"] + fact["discount_amount"]) * 100).round(2), 0)

fact["order_year"] = fact["order_date"].dt.year
fact["order_month"] = fact["order_date"].dt.month
fact["order_quarter"] = fact["order_date"].dt.quarter
fact["order_dow"] = fact["order_date"].dt.dayofweek # 0=Monday
fact["order_dow_name"] = fact["order_date"].dt.day_name()
fact["order_ym"] = fact["order_date"].dt.to_period("M").astype(str)

save(fact,"fact_orders_enriched.csv")

  Saved fact_orders_enriched.csv: 714,669 rows (181.9 MB)


In [16]:
fact_ret = returns.copy()
fact_ret = fact_ret.merge(
products[["product_id","product_name","category","segment",
"size","color","price","cogs"]],
on="product_id", how="left")
fact_ret = fact_ret.merge(
orders[["order_id","order_date","customer_id","order_source","device_type"]],
on="order_id", how="left")
fact_ret["days_to_return"] = (fact_ret["return_date"] - fact_ret["order_date"]).dt.days

save(fact_ret,"fact_returns_enriched.csv")

  Saved fact_returns_enriched.csv: 39,939 rows (6.8 MB)


In [5]:
fact_sales = sales.copy()
fact_sales.columns = ['date', 'revenue', 'cogs']
fact_sales['gross_profit']     = (fact_sales['revenue'] - fact_sales['cogs']).round(2)
fact_sales['gross_margin_pct'] = (
    fact_sales['gross_profit'] / fact_sales['revenue'] * 100).round(2)
fact_sales['year_month'] = fact_sales['date'].dt.to_period('M').astype(str)

# Rolling averages
fact_sales = fact_sales.sort_values('date')
fact_sales['revenue_ma7']  = fact_sales['revenue'].rolling(7,  min_periods=1).mean().round(2)
fact_sales['revenue_ma30'] = fact_sales['revenue'].rolling(30, min_periods=1).mean().round(2)

# YoY growth
fact_sales['revenue_ly'] = fact_sales['revenue'].shift(365)
fact_sales['yoy_growth_pct'] = np.where(
    fact_sales['revenue_ly'] > 0,
    ((fact_sales['revenue'] - fact_sales['revenue_ly']) / fact_sales['revenue_ly'] *
            100).round(2),
    np.nan)
fact_sales.drop(columns=['revenue_ly'], inplace=True)
save(fact_sales, "fact_sales.csv")

  Saved fact_sales.csv: 3,833 rows (0.3 MB)


In [8]:
monthly = fact_sales.groupby("year_month").agg(
    revenue = ("revenue", "sum"),
    cogs = ("cogs", "sum"),
    gross_profit = ("gross_profit", "sum"),
    avg_daily_rev = ("revenue", "mean"),
    days_in_month = ("revenue", "count"),
).reset_index()
monthly["revenue_prev"] = monthly["revenue"].shift(1)
monthly["mom_growth_pct"] = np.where(
 monthly["revenue_prev"] > 0,
 ((monthly["revenue"] - monthly["revenue_prev"]) / monthly["revenue_prev"] * 100).
round(2),
 np.nan)

save(monthly,"agg_monthly_summary.csv")

  Saved agg_monthly_summary.csv: 126 rows (0.0 MB)
